# BME 603/606: Computational Methods for Biological Modeling & Simulation
## In-Class Activity: Gillespie Stochastic Simulation Algorithm

Recall the Gillespie algorithm from lecture. Given current molecule counts $\mathbf{X}$ and $M$ reactions with propensities $a_j = c_j h_j$, we repeat the following steps:

1. **Total propensity:** $a_0 = \sum_{j=1}^{M} a_j$.
2. **Time to next reaction:** draw $u_1 \sim \mathrm{Unif}([0,1])$, set $\tau = -\ln(u_1) / a_0$.
3. **Which reaction:** draw $u_2 \sim \mathrm{Unif}([0,1])$, find the smallest $j$ with $\sum_{i=1}^{j} a_i > u_2 a_0$. Apply reaction $j$'s stoichiometry, set $t \leftarrow t + \tau$.

Here $h_j$ is the statistical factor (number of eligible combinations), and $c_j$ is the stochastic rate constant.

Today you'll implement this algorithm in code and use it to simulate **transcriptional bursting**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports done!")

---
## Background: Transcriptional Bursting

A gene in a single cell doesn't produce mRNA at a steady rate. It switches between an inactive state and an active state. When it's ON, transcripts are produced rapidly; when it's OFF, production stops. So mRNA arrives in bursts — a clump of transcripts, then a quiet period, then another clump. If you watch a single cell over time, you'll see these bursts as step-like jumps in the mRNA count rather than a smooth curve.

As a result mRNA is not Poisson distributed. A constant-rate production/degradation process (like we model with ODEs) gives a Poisson steady state. Bursting gives a broader, skewed distribution — more cells with zero or very high mRNA than Poisson would predict. Two genes can have the same mean mRNA level but very different noise depending on whether they produce many small bursts or few large ones. Today you'll look for these signatures in gene expression trajectories and histograms generated using the Gillespie SSA.

### The Telegraph Model

The minimal model for bursting is called the **telegraph model**, named after the on/off signaling of a telegraph: the gene toggles between an inactive state $G_{\text{off}}$ and an active state $G_{\text{on}}$, like a telegraph key being pressed and released. Transcripts are produced only when the gene is active and degrading constitutively.

| Reaction | Description | Propensity |
|---|---|---|
| $R_1$: $G_{\text{off}} \to G_{\text{on}}$ | gene activation | $k_{\text{on}} n_{G_{\text{off}}}$ |
| $R_2$: $G_{\text{on}} \to G_{\text{off}}$ | gene inactivation | $k_{\text{off}} n_{G_{\text{on}}}$ |
| $R_3$: $G_{\text{on}} \to G_{\text{on}} + M$ | transcription | $k_{\text{tx}} n_{G_{\text{on}}}$ |
| $R_4$: $M \to \varnothing$ | mRNA degradation | $k_{\text{deg}} n_M$ |

Here, every reaction is first-order in the species count, so $c_j = k_j$ for all four reactions. No $\Omega$ conversion is needed, unlike the EGFR problem (in the lecture) where the bimolecular $R_1$ and $R_3$ required $k/\Omega$ and $2k/\Omega$. The mass-action rate $k_{\text{tx}}[G_{\text{on}}]$ is already an events-per-second rate because $[G_{\text{on}}]$ is 0 or 1 divided by the volume, and multiplying by the volume gives back $n_{G_{\text{on}}}$.

Parameters:

| Parameter | Value | Interpretation |
|---|---|---|
| $k_{\text{on}}$ | $0.05\ \text{s}^{-1}$ | gene turns on once per ~20 s |
| $k_{\text{off}}$ | $0.10\ \text{s}^{-1}$ | gene turns off once per ~10 s |
| $k_{\text{tx}}$ | $0.50\ \text{s}^{-1}$ | one transcript per ~2 s while active |
| $k_{\text{deg}}$ | $0.02\ \text{s}^{-1}$ | mRNA half-life ~35 s |

Initial state: $[n_{G_{\text{off}}}, n_{G_{\text{on}}}, n_M] = [1, 0, 0]$.

The deterministic steady-state mean is $\langle M \rangle = k_{\text{tx}} \cdot k_{\text{on}} / (k_{\text{on}} + k_{\text{off}}) / k_{\text{deg}} \approx 8.3$.

---
## Part 1: Implement the Gillespie SSA

### Exercise 1: Write the Simulator

**Tasks:**
1. Complete `gillespie_ssa` below. It should take a stoichiometry matrix `nu` (one row per reaction, one column per species), a propensity function `propensities(x)`, initial counts, and the number of steps.
2. Return three arrays: times `t` (length `n_steps + 1`), states `X` (shape `(n_steps + 1, n_species)`), and reaction indices `which` (length `n_steps`).
3. Write `props_burst(x)` and `nu_burst` for the telegraph model. Run three events with `rng=np.random.default_rng(0)` and print your $\tau$ and which reaction fired at each step. Sanity-check that the propensities look right at each state.

> **Hint:** `np.searchsorted(np.cumsum(a), u2 * a0, side='right')` gives you $j$ directly. If $a_0 = 0$ (all propensities zero), break out of the loop.

In [ ]:
def gillespie_ssa(nu, propensities, x0, n_steps, rng=None):
    """
    Gillespie Stochastic Simulation Algorithm.

    Parameters
    ----------
    nu : array, shape (M, n_species)
        Stoichiometry matrix. nu[j] is the change vector when reaction j fires.
    propensities : callable
        propensities(x) -> array of length M with a_j at state x.
    x0 : array, shape (n_species,)
        Initial molecule counts.
    n_steps : int
        Number of reaction events to simulate.
    rng : np.random.Generator or None
        Random number generator; if None, use a fresh default_rng().

    Returns
    -------
    t     : array, shape (n_steps + 1,) — time at each recorded state
    X     : array, shape (n_steps + 1, n_species) — molecule counts
    which : array, shape (n_steps,) — index of reaction that fired at each step
    """
    if rng is None:
        rng = np.random.default_rng()

    n_species = len(x0)
    t = np.zeros(n_steps + 1)
    X = np.zeros((n_steps + 1, n_species), dtype=int)
    which = np.zeros(n_steps, dtype=int)
    X[0] = x0

    # Your code here — loop over steps, compute a0, draw u1, u2, update
    pass

    return t, X, which


# ── Transcriptional bursting model ───────────────────────────────────────────
# Species order: [G_off, G_on, M]
nu_burst = np.array([
    [-1, +1,  0],   # R1: Goff -> Gon
    [+1, -1,  0],   # R2: Gon -> Goff
    [ 0,  0, +1],   # R3: Gon -> Gon + M
    [ 0,  0, -1],   # R4: M -> 0
])

k_on, k_off, k_tx, k_deg = 0.05, 0.10, 0.50, 0.02

def props_burst(x):
    # Your code here — return the length-4 propensity array
    pass

x0_burst = np.array([1, 0, 0])

# Your code here — run 3 steps at seed 0, print tau and reaction at each step

### Exercise 2: A Single Trajectory — See the Bursts

**Tasks:**
1. Simulate 2000 events from the bursting model with `rng=np.random.default_rng(0)`. Plot $[M](t)$ — the mRNA count vs. time.
2. On a second axis below, plot the gene state $[G_{\text{on}}]$ as a step function. Shade the times when the gene is ON.
3. **Interpret:** Do transcripts arrive uniformly in time, or in clumps? How does each ON interval correspond to a burst of mRNA?

> **Hint:** For shading, `plt.axvspan(t[i], t[i+1], alpha=0.1, color='C0')` inside a loop where `X[i, 1] == 1`.

In [ ]:
# Your code here — simulate and plot M(t) with gene-state shading

### Exercise 3: Seed-to-Seed Variability

**Tasks:**
1. Run the 2000-event bursting simulation for seeds `0, 1, 2, 3, 4`. Plot $[M](t)$ for all five on the same axes (no shading — this one is about variability).
2. Record the final $[M]$ for each seed. How much does it spread?
3. **Interpret:** If you ran 1000 trajectories, what would you expect the *distribution* of final $[M]$ to look like?

In [ ]:
# Your code here — M(t) for seeds 0..4, plus the final M for each

### Exercise 4: The Distribution of mRNA

Bursting produces a non-Poisson steady-state distribution of mRNA. You can see it directly by running many trajectories and histogramming the mRNA count at a fixed observation time.

**Tasks:**
1. Run 1000 independent trajectories each for 3000 events. Record $[M]$ at $t = 1000\ \text{s}$ (interpolating if the trajectory doesn't land exactly at $t = 1000$).
2. Plot the histogram of $[M]$ at that time. Overlay a Poisson distribution with the same mean.
3. **Interpret:** Does the histogram look Poisson? If not, what does that tell you about the underlying process? (Compare to the deterministic ODE — would the ODE predict any variance at all?)

> **Hint:** For each trajectory, find the last time $t_i \le 1000$ with `np.searchsorted(t, 1000, side='right') - 1` and record `X[idx, 2]`. To overlay a Poisson, use `from scipy.stats import poisson` and `poisson.pmf(np.arange(...), mu)`.

In [ ]:
from scipy.stats import poisson

# Your code here — 1000 trajectories, histogram M at t=1000, overlay Poisson

### Exercise 5: Tune the Burst Size and Frequency

The two key bursting parameters are the **burst frequency** ($\approx k_{\text{on}}$, how often the gene turns on) and the **burst size** ($\approx k_{\text{tx}}/k_{\text{off}}$, how many transcripts per ON interval). Together, they set the mean via $\langle M \rangle \approx \text{frequency} \times \text{burst size} / k_{\text{deg}}$.

**Tasks:**
1. Fix the mean at $\langle M \rangle \approx 8.3$ by keeping $k_{\text{on}} k_{\text{tx}} / (k_{\text{off}} k_{\text{deg}})$ constant. Vary $k_{\text{on}}$ from $0.01$ to $0.5\ \text{s}^{-1}$ while adjusting $k_{\text{tx}}$ to keep the mean fixed.
2. For each parameter set, run 2000 trajectories, record $[M]$ at $t = 1000$, and plot the **histogram** as a row of a heatmap (x = mRNA count, y = $k_{\text{on}}$).
3. **Interpret:** At low $k_{\text{on}}$ (rare, large bursts), what does the distribution look like? At high $k_{\text{on}}$ (frequent, small bursts), what does it look like? Why does the shape change?

> **Hint:** If you keep $\langle M \rangle$ fixed at 8.3, then $k_{\text{tx}} = \langle M \rangle \cdot k_{\text{deg}} \cdot (k_{\text{on}} + k_{\text{off}}) / k_{\text{on}}$. Plot with `plt.imshow(..., aspect='auto', origin='lower')`.

In [ ]:
# Your code here — sweep k_on, adjust k_tx, collect histograms, plot heatmap

---
## Part 2: Challenge Exercise (Bonus)

### Exercise 6: Tau Leaping

**For bonus credit.** The Gillespie SSA is slow when propensities are large: each event advances the clock by $\tau \sim 1/a_0$, so simulating 1000 s of a high-expression gene takes millions of steps. **Tau leaping** jumps a fixed $\tau$ and samples how many of each reaction fired in that interval.

**The approximation:** in a time interval $\tau$, the number of firings of reaction $j$ is $\mathrm{Poisson}(a_j \tau)$, provided $\tau$ is small enough that the propensities don't change much.

**The algorithm:**
1. Compute $a_j(\mathbf{X})$ at current state.
2. For each $j$: draw $k_j \sim \mathrm{Poisson}(a_j \tau)$.
3. Update $\mathbf{X} \leftarrow \mathbf{X} + \sum_j k_j \nu_j$, with clamping to prevent negative counts.
4. $t \leftarrow t + \tau$.

### Exercise 7: Implement Tau Leaping

**Tasks:**
1. Implement `tau_leap_burst` below. Clamp each $k_j$ so it can't drive a species below zero — for this network, $k_1 \le n_{G_{\text{off}}}$, $k_2 \le n_{G_{\text{on}}}$, $k_4 \le n_M$, and $k_3$ must be evaluated *before* $R_1$ and $R_2$ fire
2. Run for 200 steps at $\tau = 1\ \text{s}$, $5\ \text{s}$, and $20\ \text{s}$. Overlay the resulting $[M](t)$ against an SSA trajectory on the same axes.
3. **Interpret:** Which $\tau$ stays close to the SSA trajectory? Which one diverges? Why does the gene-state switching rate $k_{\text{on}} + k_{\text{off}} \approx 0.15\ \text{s}^{-1}$ impose a constraint on $\tau$?

> **Hint:** The binding constraint is $a_j \tau \ll \mathbf{X}$, i.e. the *relative* change in any species must be small. For the gene, $\mathbf{X} \in \{0, 1\}$, so you need $(k_{\text{on}} + k_{\text{off}})\tau \ll 1$, i.e. $\tau \ll 6.7\ \text{s}$. Large $\tau$ lets the gene flip multiple times per step, which the Poisson draw cannot capture — the naive tau leap over-counts transcription because it treats the ON state as frozen.

In [ ]:
def tau_leap_burst(k_on, k_off, k_tx, k_deg, tau, n_steps, rng=None):
    """
    Tau-leaping approximation of the transcriptional bursting model.

    Parameters
    ----------
    k_on, k_off, k_tx, k_deg : float — rate constants
    tau : float — fixed leap size
    n_steps : int — number of leaps
    rng : np.random.Generator or None

    Returns
    -------
    t : array, shape (n_steps + 1,)
    X : array, shape (n_steps + 1, 3) — [G_off, G_on, M]
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.array([1, 0, 0])
    t = np.zeros(n_steps + 1)
    X = np.zeros((n_steps + 1, 3), dtype=int)
    X[0] = x

    # Your code here — for each step: compute a, draw Poisson, clamp, update
    pass

    return t, X


# Your code here — run for tau = 1, 5, 20; overlay against SSA

### Exercise 8: When Does Tau Leaping Pay Off?

**Tasks:**
1. Time the SSA and tau leaping on a **high-expression** version of the model. Multiply $k_{\text{tx}}$ by 100 (so $\langle M \rangle \approx 830$), and simulate 500 s of biological time.
2. For the SSA, time 500 s of simulation. For tau leaping, use $\tau = 0.5\ \text{s}$ and time 1000 steps (which is also 500 s).
3. Compare the wall-clock time and the mean $\langle M \rangle$ over the last 100 s of each.
4. **Interpret:** Does tau leaping give you a speedup here? What does the speedup depend on — the number of events, the number of leap steps, or something else? When would tau leaping *not* be worth using?

> **Hint:** `import time; t0 = time.perf_counter(); ...; print(time.perf_counter() - t0)`. The SSA will be slow enough that you'll want to interrupt it if it runs more than ~30 s — time it in a `try/except KeyboardInterrupt` if you like.

In [ ]:
import time

# Your code here — timing comparison at k_tx = 50 (100x the baseline)

**Your interpretation of SSA vs. tau leaping:**

*Write your answer here. Under what conditions is the Gillespie SSA worth its cost? When would you reach for tau leaping instead? What does the answer have to do with the biological system you're modeling — a low-copy-number gene versus a high-abundance metabolic enzyme?*

---
## Further Exploration

### The SSA Literature

Gillespie's 1977 paper, "Exact stochastic simulation of coupled chemical reactions" (*J. Phys. Chem.* 81, 2340), is the canonical reference. It's remarkably readable and worth skimming for the master-equation derivation.

### The Telegraph Model and Bursting

The two-state telegraph model you simulated is the standard minimal model of transcriptional bursting. Peccoud & Ycart (1995) derived its exact steady-state distribution — a confluent hypergeometric, not a Poisson, which is why your histograms in Exercise 4 looked skewed. For a review of bursting in real cells, see Tunnacliffe & Chubb (2020), *"What is a transcriptional burst?"* (*Trends Genet.* 36, 288).

### Adaptive Tau Leaping

The fixed-$\tau$ scheme you implemented is the simplest version. The **Cao–Gillespie–Petzold** adaptive method (2006) chooses $\tau$ at each step to bound the relative change in every propensity by a parameter $\epsilon$ — the same idea as the "choose $\tau$ so $a_j \tau \ll \mathbf{X}$" heuristic you saw, but automated. This is what production codes use.

### References & Resources

- [Gillespie algorithm — Wikipedia](https://en.wikipedia.org/wiki/Gillespie_algorithm)
- [Gillespie, D. T. (1977). Exact stochastic simulation of coupled chemical reactions. *J. Phys. Chem.* 81, 2340.](https://pubs.acs.org/doi/10.1021/j100540a008)
- [Cao, Gillespie, Petzold (2006). Efficient step size selection for the tau-leaping simulation method. *J. Chem. Phys.* 124, 044109.](https://aip.scitation.org/doi/10.1063/1.2159468)
- [Lewis Coleman's blog post on the Gillespie algorithm](https://lewiscoleblog.com/gillespie-algorithm)
- [SciPy `scipy.stats.poisson`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.poisson.html)
- [NumPy `Generator` documentation](https://numpy.org/doc/stable/reference/random/generator.html)